=============================================================================
VeritasLens: Real-Time Hallucination Detection with Gemma 4
=============================================================================
Gemma 4 Good Hackathon | Google DeepMind x Kaggle
Track: Safety & Trust | Main Track | Special: Unsloth

Author: Jacob O. | UCSD Physics & Data Science
        Incoming Berkeley MIDS 2026

What this notebook does end to end:
  1. Installs dependencies
  2. Loads Gemma 4 E4B via Unsloth
  3. Defines a curated offline hallucination knowledge base
  4. Implements structured evidence retrieval that ACTUALLY fires and logs
  5. Fine-tunes on hallucination detection data via Unsloth QLoRA
  6. Runs evaluation suite (held-out cases not in training data)
  7. Shows interpretability: which words most influence each verdict
  8. Exports GGUF for local Ollama deployment
  9. Launches Gradio demo with live results

Prize tracks: Safety & Trust + Main Track + Unsloth
=============================================================================

In [1]:
# =============================================================================
# CELL 1 - Install
# =============================================================================

import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "trl", "datasets", "gradio", "requests"
])

print("Installation complete.")

Installation complete.


In [2]:
# =============================================================================
# CELL 2 - Imports and GPU check
# =============================================================================

import os
import re
import sys
import json
import time
import random
import hashlib
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import urllib.parse

import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: {:.1f} GB".format(
        torch.cuda.get_device_properties(0).total_memory / 1e9
    ))

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# =============================================================================
# CELL 3 - Load Gemma 4 E4B via Unsloth
# =============================================================================

from unsloth import FastModel

MODEL_NAME = "unsloth/gemma-4-E4B-it"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,
    max_seq_length=8192,
    load_in_4bit=True,
    full_finetuning=False,
)

print("Model loaded:", MODEL_NAME)
print("Parameters:", sum(p.numel() for p in model.parameters()))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Model loaded: unsloth/gemma-4-E4B-it
Parameters: 5979286048


In [4]:
# =============================================================================
# CELL 4 - Chat template
# =============================================================================

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Get the raw HuggingFace tokenizer so we can call apply_chat_template
# on plain string content without the processor's list-of-dicts requirement
raw_tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

print("Chat template: gemma-4")

Chat template: gemma-4


In [5]:
# =============================================================================
# CELL 5 - Offline knowledge base
# =============================================================================
HALLUCINATION_KB = {
    "great wall visible from space": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "The Great Wall of China is NOT visible from space with the naked eye. "
            "The wall is only about 9 meters wide, far too narrow to resolve from "
            "orbital altitude. Confirmed by NASA and China's own astronaut Yang Liwei."
        ),
        "source": "NASA; China National Space Administration"
    },
    "einstein failed math": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "Einstein did not fail math. He excelled at mathematics from an early age "
            "and had mastered calculus by age 15. This myth arose from misreading "
            "Swiss school grade scales where 6 is the highest mark, not the lowest."
        ),
        "source": "ETH Zurich historical records; Einstein Archive"
    },
    "humans use 10 percent brain": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "Humans use virtually all regions of their brain. Brain imaging studies "
            "show no consistently dormant 90 percent region. This myth has no basis "
            "in neuroscience."
        ),
        "source": "Johns Hopkins Medicine; Scientific American"
    },
    "napoleon was short": {
        "verdict": "CONTESTED",
        "correct": (
            "Napoleon was approximately 5 feet 7 inches (170 cm), average for his era. "
            "The short myth originated from British propaganda and confusion between "
            "French and English inch measurements."
        ),
        "source": "Historical records; Marengo exhibition"
    },
    "curie three nobel": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "Marie Curie won exactly two Nobel Prizes: Physics in 1903 and "
            "Chemistry in 1911. No third prize exists in the record."
        ),
        "source": "Nobel Prize Foundation official records"
    },
    "vitamin c prevents colds": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "Vitamin C does not prevent colds in the general population. "
            "The Cochrane Review analyzed 31 trials with over 11,000 participants "
            "and found no reduction in cold incidence."
        ),
        "source": "Cochrane Review 2023; NIH Office of Dietary Supplements"
    },
    "amazon longest river": {
        "verdict": "CONTESTED",
        "correct": (
            "Whether the Amazon or Nile is longer is disputed. The Nile is traditionally "
            "measured as longer at about 6,650 km. A 2008 Brazilian study contested this. "
            "The Amazon does carry more freshwater than any other river."
        ),
        "source": "USGS; INPE Brazil 2008"
    },
    "einstein definition insanity": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "There is no verified Einstein source for this quote. "
            "Quote Investigator traces its earliest known appearance to "
            "Narcotics Anonymous literature in 1981, decades after Einstein died."
        ),
        "source": "Quote Investigator; Narcotics Anonymous archives"
    },
    "goldfish 3 second memory": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "Goldfish have memory spans of at least three months. "
            "Researchers at Plymouth University trained goldfish to navigate mazes."
        ),
        "source": "Plymouth University research 2003"
    },
    "lightning never strikes twice": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "Lightning frequently strikes the same location multiple times. "
            "The Empire State Building is struck approximately 20 to 25 times per year."
        ),
        "source": "NOAA; National Lightning Safety Institute"
    },
    "bulls hate red": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "Bulls are red-green colorblind. They react to the movement of the "
            "matador's cape, not its color. A white cape produces the same response."
        ),
        "source": "Animal behavior research; Scientific American"
    },
    "humans swallow spiders sleeping": {
        "verdict": "UNSUPPORTED",
        "correct": (
            "The claim that humans swallow 8 spiders per year in their sleep is false. "
            "Spiders avoid the vibrations, breath, and heat of sleeping humans."
        ),
        "source": "Entomology research; Snopes"
    }
}

# FIX: entity registry keys now looked up case-insensitively via helper
_ENTITY_REGISTRY_RAW = {
    ("albert einstein", "person"): {
        "born": 1879, "died": 1955,
        "nationality": "German-Swiss-American",
        "field": "Physics",
        "nobel_prizes": 1,
        "failed_math": False,
        "mastered_calculus_age": 15
    },
    ("marie curie", "person"): {
        "born": 1867, "died": 1934,
        "nationality": "Polish-French",
        "nobel_prizes": 2,
        "fields": ["Physics", "Chemistry"],
        "first_female_nobel": True
    },
    ("napoleon", "person"): {
        "born": 1769, "died": 1821,
        "height_cm": 170,
        "title": "Emperor of the French"
    },
    ("great wall of china", "place"): {
        "length_km": 21196,
        "width_meters": 9,
        "visible_from_space": False
    },
    ("amazon river", "place"): {
        "most_freshwater_volume": True,
        "traditionally_longest": False,
        "length_km_approx": 6400
    }
}

def _lookup_entity(name, entity_type):
    """Case-insensitive entity lookup with partial name matching."""
    name_lower = name.lower().strip()
    # Exact match first
    key = (name_lower, entity_type)
    if key in _ENTITY_REGISTRY_RAW:
        return _ENTITY_REGISTRY_RAW[key]
    # Partial match: name is contained in or contains a registry key
    for (k_name, k_type), v in _ENTITY_REGISTRY_RAW.items():
        if k_type == entity_type and (name_lower in k_name or k_name in name_lower):
            return v
    return {}

ENTITY_REGISTRY = _ENTITY_REGISTRY_RAW  # kept for compatibility

EVIDENCE_LOG = []

def tool_fetch_evidence(claim, search_query):
    """
    Evidence retrieval with three-tier fallback:
    1. Offline hallucination KB (instant, 12 curated entries)
    2. Wikipedia REST API (free, no key, covers millions of topics)
    3. Brave Search (if BRAVE_API_KEY set)
    Logs every call so outputs show real tool activity.
    """
    normalized = claim.lower()
    result = None

    # Tier 1: Offline KB
    for kb_key, kb_val in HALLUCINATION_KB.items():
        words = kb_key.split()
        hits = sum(1 for w in words if w in normalized and len(w) > 3)
        if hits >= 2:
            result = {
                "found": True,
                "verdict_signal": kb_val["verdict"],
                "snippet": kb_val["correct"],
                "sources": [kb_val["source"]],
                "method": "offline_knowledge_base"
            }
            break

    # Tier 2: Wikipedia REST API (free, generalizable)
    if result is None:
        wiki = _wiki_search(search_query)
        if wiki and wiki["snippet"]:
            result = {
                "found": True,
                "verdict_signal": "NEEDS_REVIEW",
                "snippet": wiki["snippet"],
                "sources": [wiki["url"] or "Wikipedia"],
                "method": "wikipedia"
            }

    # Tier 3: Brave Search
    if result is None:
        api_key = os.environ.get("BRAVE_API_KEY", "")
        if api_key:
            try:
                import requests
                resp = requests.get(
                    "https://api.search.brave.com/res/v1/web/search",
                    headers={"Accept": "application/json", "X-Subscription-Token": api_key},
                    params={"q": search_query, "count": 3},
                    timeout=5
                )
                if resp.status_code == 200:
                    results = resp.json().get("web", {}).get("results", [])
                    if results:
                        result = {
                            "found": True,
                            "verdict_signal": "NEEDS_REVIEW",
                            "snippet": results[0].get("description", "")[:300],
                            "sources": [r.get("url", "") for r in results[:3]],
                            "method": "brave_search"
                        }
            except Exception:
                pass

    if result is None:
        result = {
            "found": False,
            "verdict_signal": "UNVERIFIABLE",
            "snippet": "No evidence found.",
            "sources": [],
            "method": "offline_fallback"
        }

    EVIDENCE_LOG.append({
        "tool": "fetch_evidence",
        "claim": claim[:60],
        "verdict_signal": result["verdict_signal"],
        "method": result["method"]
    })
    return result

def tool_check_knowledge_base(claim, domain="general"):
    normalized = claim.lower()
    for kb_key, kb_val in HALLUCINATION_KB.items():
        words = kb_key.split()
        hits = sum(1 for w in words if w in normalized and len(w) > 3)
        if hits >= 2:
            result = {
                "kb_hit": True,
                "verdict": kb_val["verdict"],
                "correct_information": kb_val["correct"],
                "source": kb_val["source"],
                "domain": domain
            }
            EVIDENCE_LOG.append({
                "tool": "check_knowledge_base",
                "claim": claim[:60],
                "verdict": kb_val["verdict"],
                "domain": domain
            })
            return result
    EVIDENCE_LOG.append({
        "tool": "check_knowledge_base",
        "claim": claim[:60],
        "verdict": "NOT_IN_KB",
        "domain": domain
    })
    return {
        "kb_hit": False,
        "verdict": "NOT_IN_KB",
        "note": "Claim not in offline KB. Use fetch_evidence.",
        "domain": domain
    }

def tool_resolve_entity(entity_name, entity_type, claimed_attribute=""):
    # FIX: use the case-insensitive helper instead of direct dict lookup
    data = _lookup_entity(entity_name, entity_type)
    result = {
        "entity": entity_name,
        "found": bool(data),
        "known_facts": data if data else {}
    }
    EVIDENCE_LOG.append({
        "tool": "resolve_entity",
        "entity": entity_name,
        "found": bool(data)
    })
    return result

TOOL_DISPATCH = {
    "fetch_evidence": tool_fetch_evidence,
    "check_knowledge_base": tool_check_knowledge_base,
    "resolve_entity": tool_resolve_entity
}

def execute_tool(tool_name, tool_args):
    fn = TOOL_DISPATCH.get(tool_name)
    if fn is None:
        return {"error": "Unknown tool: {}".format(tool_name)}
    try:
        return fn(**tool_args)
    except Exception as e:
        return {"error": str(e), "tool": tool_name}

print("Knowledge base entries:", len(HALLUCINATION_KB))
print("Entity registry entries:", len(_ENTITY_REGISTRY_RAW))
print("Tool implementations ready.")

Knowledge base entries: 12
Entity registry entries: 5
Tool implementations ready.


In [6]:
# =============================================================================
# CELL 6 - System prompt
# =============================================================================
SYSTEM_PROMPT = (
    "You are VeritasLens, a precise factual grounding assistant. "
    "Your job: split the input into individual sentences, identify every "
    "verifiable factual claim, check each one, and return a structured JSON verdict.\n\n"
    "CRITICAL: Treat each sentence as a SEPARATE claim. Never combine multiple "
    "sentences into one claim entry. If the input has 3 sentences, you must "
    "produce at least 3 claim entries in your final JSON.\n\n"
    "You have three tools. Call them by outputting JSON inside <tool_call> tags.\n\n"
    "TOOL 1: fetch_evidence\n"
    "  Use for every claim before assigning a verdict.\n"
    "  Call: <tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
    "{\"claim\": \"exact claim text\", \"search_query\": \"3-6 keyword query\"}}</tool_call>\n\n"
    "TOOL 2: check_knowledge_base\n"
    "  Use to check common hallucination patterns.\n"
    "  Call: <tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
    "{\"claim\": \"exact claim\", \"domain\": \"science\"}}</tool_call>\n"
    "  Domain options: science, history, statistics, quotes, geography, medicine\n\n"
    "TOOL 3: resolve_entity\n"
    "  Use for claims about named people, places, or organizations.\n"
    "  Call: <tool_call>{\"name\": \"resolve_entity\", \"arguments\": "
    "{\"entity_name\": \"Marie Curie\", \"entity_type\": \"person\", "
    "\"claimed_attribute\": \"number of Nobel prizes\"}}</tool_call>\n"
    "  Entity type options: person, organization, place, event\n\n"
    "WORKFLOW:\n"
    "1. Split the input text into individual sentences.\n"
    "2. For EACH sentence that contains a verifiable claim: call fetch_evidence "
    "AND check_knowledge_base.\n"
    "3. For sentences about named entities, also call resolve_entity.\n"
    "4. After ALL tool calls are complete, output ONLY the final JSON verdict "
    "with no other text before or after it.\n\n"
    "FINAL OUTPUT FORMAT (output this after all tool calls, NOTHING ELSE):\n"
    "{\"claims\": [{\"text\": \"exact sentence\", \"verdict\": \"SUPPORTED or UNSUPPORTED "
    "or CONTESTED or UNVERIFIABLE or OPINION\", \"confidence\": 0.0-1.0, "
    "\"evidence\": [\"source1\"], \"reasoning\": \"one sentence\", "
    "\"corrected\": \"corrected text or null\"}], "
    "\"overall_reliability\": 0.0-1.0, \"summary\": \"two sentence summary\"}\n\n"
    "VERDICT DEFINITIONS:\n"
    "SUPPORTED: claim is accurate per evidence\n"
    "UNSUPPORTED: claim is false or not supported by evidence\n"
    "CONTESTED: sources genuinely disagree\n"
    "UNVERIFIABLE: cannot determine with available evidence\n"
    "OPINION: subjective statement, not a factual claim\n\n"
    "Always provide a corrected version for every UNSUPPORTED claim. "
    "Output valid JSON only. Do not wrap it in markdown code blocks."
)

print("System prompt length:", len(SYSTEM_PROMPT), "characters")

System prompt length: 2400 characters


In [7]:
# =============================================================================
# CELL 7 - Inference with tool parsing
# =============================================================================
import urllib.request

def _wiki_search(query, max_chars=400):
    """Free Wikipedia REST API -- no key required."""
    try:
        q = urllib.parse.quote(query[:100])
        url = "https://en.wikipedia.org/api/rest_v1/page/summary/{}".format(q)
        req = urllib.request.Request(url, headers={"User-Agent": "VeritasLens/1.0"})
        with urllib.request.urlopen(req, timeout=5) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                extract  = data.get("extract", "")
                page_url = data.get("content_urls", {}).get("desktop", {}).get("page", "")
                if extract:
                    return {"snippet": extract[:max_chars], "url": page_url}
    except Exception:
        pass
    return None

def parse_tool_calls(text):
    calls = []
    for match in re.finditer(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL):
        raw = match.group(1).strip()
        try:
            parsed = json.loads(raw)
            name = parsed.get("name") or parsed.get("function", {}).get("name")
            args = parsed.get("arguments") or parsed.get("parameters") or {}
            if name:
                calls.append((name, args))
        except (json.JSONDecodeError, AttributeError):
            continue
    return calls

def extract_json_from_text(text):
    clean = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.DOTALL)
    for pattern in [r"```json\n(.*?)```", r"```\n(.*?)```", r"```json(.*?)```"]:
        m = re.search(pattern, clean, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1).strip())
            except json.JSONDecodeError:
                pass
    depth, start = 0, None
    for i, ch in enumerate(clean):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                candidate = clean[start:i+1]
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    start = None
    return None

def _generate_response(current_messages, max_new_tokens=1500):
    prompt = raw_tok.apply_chat_template(
        current_messages,
        add_generation_prompt=True,
        tokenize=False
    )
    inputs   = raw_tok(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            use_cache=True
        )
    return raw_tok.decode(
        output_ids[0][input_len:],
        skip_special_tokens=True
    ).strip()

def _has_valid_claims(response):
    """
    Returns True only if the response contains parseable JSON
    with at least one claim whose verdict is a known non-UNVERIFIABLE value.
    A model that outputs {"claims": [{"verdict": "UNVERIFIABLE"}]} is NOT
    considered to have done valid grounding work.
    """
    j = extract_json_from_text(response)
    if not j or "claims" not in j:
        return False
    claims = j["claims"]
    if not claims:
        return False
    decisive = {"SUPPORTED", "UNSUPPORTED", "CONTESTED", "OPINION"}
    return any(c.get("verdict") in decisive for c in claims)

def run_inference(messages, max_tool_rounds=10, max_new_tokens=1500):
    """
    Multi-round tool-calling loop with guaranteed final JSON pass.

    Regression guard logic (round 0):
    - If the model emits tool calls -> proceed normally (good path)
    - If the model emits valid grounded JSON directly -> accept it
    - If the model emits JSON but all verdicts are UNVERIFIABLE (post-SFT
      regression) OR emits no JSON at all -> inject nudge and retry TWICE
      before falling through to forced JSON pass
    """
    EVIDENCE_LOG.clear()
    current_messages = []
    for msg in messages:
        content = msg["content"]
        if isinstance(content, list):
            content = " ".join(
                part.get("text", "") for part in content if isinstance(part, dict)
            )
        current_messages.append({"role": msg["role"], "content": content})

    FastModel.for_inference(model)
    nudge_count = 0

    for round_num in range(max_tool_rounds):
        response   = _generate_response(current_messages, max_new_tokens)
        tool_calls = parse_tool_calls(response)

        # --- REGRESSION GUARD ---
        if round_num == 0 and not tool_calls:
            if _has_valid_claims(response):
                # Model produced valid grounded JSON directly -- accept it
                return response
            # Model produced weak/empty/all-UNVERIFIABLE output -- nudge
            current_messages.append({"role": "assistant", "content": response})
            current_messages.append({
                "role": "user",
                "content": (
                    "You did not call any evidence tools. This is required. "
                    "You MUST call fetch_evidence AND check_knowledge_base for EACH claim "
                    "before outputting a verdict. Do NOT output JSON yet. "
                    "Start by calling the tools now."
                )
            })
            nudge_count += 1
            continue

        # Second nudge: if round 1 still produced no tool calls after first nudge
        if nudge_count > 0 and round_num == 1 and not tool_calls:
            if _has_valid_claims(response):
                return response
            current_messages.append({"role": "assistant", "content": response})
            current_messages.append({
                "role": "user",
                "content": (
                    "REMINDER: Call fetch_evidence with the claim text and a search query. "
                    "Example: <tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
                    "{\"claim\": \"the claim text here\", \"search_query\": \"keywords\"}}</tool_call> "
                    "Do this for each claim NOW."
                )
            })
            continue

        if not tool_calls:
            return response

        current_messages.append({"role": "assistant", "content": response})
        tool_results = []
        for tool_name, tool_args in tool_calls:
            result = execute_tool(tool_name, tool_args)
            tool_results.append(
                "Tool: {} | Args: {} | Result: {}".format(
                    tool_name,
                    json.dumps(tool_args)[:120],
                    json.dumps(result)[:400]
                )
            )
        current_messages.append({
            "role": "user",
            "content": (
                "Tool results:\n" + "\n".join(tool_results) +
                "\n\nAll tool calls are complete. "
                "Now output ONLY the final JSON verdict. "
                "Do not call any more tools. "
                "Output valid JSON starting with { and ending with }. "
                "No markdown, no prose, no explanation."
            )
        })

    print("Warning: max_tool_rounds reached, forcing final JSON generation.")
    current_messages.append({
        "role": "user",
        "content": (
            "FINAL STEP: Output ONLY the JSON verdict now. "
            "Start with { and end with }. No tool calls. No markdown. "
            "Include all claims analyzed so far."
        )
    })
    return _generate_response(current_messages, max_new_tokens=2000)

print("Inference helpers ready.")

Inference helpers ready.


In [8]:
# =============================================================================
# CELL 8 - Main grounding function
# =============================================================================
import hashlib, time
from dataclasses import dataclass, field

VERDICT_WEIGHTS = {
    "SUPPORTED": 1.0,
    "UNSUPPORTED": 0.0,
    "CONTESTED": 0.5,
    "UNVERIFIABLE": 0.3,
    "OPINION": 0.5,
}

@dataclass
class GroundingResult:
    input_text: str
    claims: list = field(default_factory=list)
    overall_reliability: float = 0.5
    tools_called: list = field(default_factory=list)
    processing_time_s: float = 0.0
    session_id: str = ""
    raw_response: str = ""

    def summary(self):
        unsupported = [c for c in self.claims if c.get("verdict") == "UNSUPPORTED"]
        supported   = [c for c in self.claims if c.get("verdict") == "SUPPORTED"]
        contested   = [c for c in self.claims if c.get("verdict") == "CONTESTED"]
        lines = [
            "",
            "=" * 62,
            " VeritasLens Analysis",
            "=" * 62,
            " Overall reliability : {:.0%}".format(self.overall_reliability),
            " Claims analyzed     : {}".format(len(self.claims)),
            " Supported           : {}".format(len(supported)),
            " Not supported       : {}".format(len(unsupported)),
            " Contested           : {}".format(len(contested)),
            " Evidence calls made : {}".format(len(self.tools_called)),
            " Processing time     : {:.1f}s".format(self.processing_time_s),
            "-" * 62
        ]
        if self.tools_called:
            lines.append(" Evidence retrieved:")
            for call in self.tools_called[:8]:
                lines.append("  [{}] {} -> {}".format(
                    call.get("tool", ""),
                    call.get("claim", call.get("entity", ""))[:40],
                    call.get("verdict_signal", call.get("verdict", ""))
                ))
        lines.append("-" * 62)
        for i, claim in enumerate(self.claims, 1):
            verdict = claim.get("verdict", "UNKNOWN")
            conf    = claim.get("confidence", 0)
            text    = claim.get("text", "")[:70]
            marker  = {"SUPPORTED": "OK", "UNSUPPORTED": "!!", "CONTESTED": "??",
                       "UNVERIFIABLE": "--", "OPINION": "~~"}.get(verdict, " ")
            lines.append(" [{}] ({:.0%}) {}".format(marker, conf, text))
            if claim.get("corrected"):
                lines.append("      -> {}".format(claim["corrected"][:70]))
        lines.append("=" * 62)
        return "\n".join(lines)


def _compute_reliability(claims):
    """
    Compute overall reliability from verdicts directly.
    Never trust the model's self-reported overall_reliability score.
    Verifiable claims only (not OPINION/UNVERIFIABLE).
    """
    verifiable = [c for c in claims if c.get("verdict") in ("SUPPORTED", "UNSUPPORTED", "CONTESTED")]
    if not verifiable:
        return 0.5
    return round(
        sum(VERDICT_WEIGHTS.get(c["verdict"], 0.5) for c in verifiable) / len(verifiable),
        2
    )


def _split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def build_user_message(input_text):
    sentences = _split_sentences(input_text)
    if len(sentences) > 1:
        numbered = "\n".join("{}. {}".format(i+1, s) for i, s in enumerate(sentences))
        return (
            "Analyze each of the following {} sentences SEPARATELY for factual accuracy. "
            "Each sentence must become its own claim entry in your JSON output.\n\n"
            "Sentences to analyze:\n{}\n\n"
            "For each sentence: call fetch_evidence, call check_knowledge_base, "
            "and call resolve_entity for any named people or places. "
            "Then output your JSON verdict with one claim object per sentence."
        ).format(len(sentences), numbered)
    else:
        return (
            "Analyze the following text for factual accuracy. "
            "For each verifiable claim: call fetch_evidence, call check_knowledge_base, "
            "and call resolve_entity for any named people or places. "
            "Then output your JSON verdict.\n\nText to analyze:\n\n" + input_text
        )


def ground_text(input_text, verbose=True):
    session_id = hashlib.md5(input_text.encode()).hexdigest()[:8]
    start = time.time()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_message(input_text)}
    ]
    if verbose:
        print("Analyzing: {}...".format(input_text[:65]))

    response = run_inference(messages)
    result_json = extract_json_from_text(response)
    elapsed = time.time() - start

    if result_json and "claims" in result_json:
        claims = result_json.get("claims", [])
        # FIX: always recompute reliability from actual verdicts
        reliability = _compute_reliability(claims)
    else:
        sentences = _split_sentences(input_text)
        claims = []
        for sent in sentences:
            sent_lower = sent.lower()
            matched_verdict = "UNVERIFIABLE"
            for entry in EVIDENCE_LOG:
                claim_fragment = entry.get("claim", "").lower()
                if any(w in sent_lower for w in claim_fragment.split() if len(w) > 3):
                    v = entry.get("verdict_signal") or entry.get("verdict", "")
                    if v in ("SUPPORTED", "UNSUPPORTED", "CONTESTED"):
                        matched_verdict = v
                        break
            claims.append({
                "text": sent,
                "verdict": matched_verdict,
                "confidence": 0.6 if matched_verdict != "UNVERIFIABLE" else 0.3,
                "evidence": [],
                "reasoning": "Verdict inferred from KB evidence log (JSON parse failed).",
                "corrected": None
            })
        reliability = _compute_reliability(claims)
        if verbose:
            print("Warning: model did not return parseable JSON. Used KB fallback.")
            print("Raw response snippet:", response[:300])

    result = GroundingResult(
        input_text=input_text,
        claims=claims,
        overall_reliability=reliability,
        tools_called=list(EVIDENCE_LOG),
        processing_time_s=round(elapsed, 2),
        session_id=session_id,
        raw_response=response
    )
    if verbose:
        print(result.summary())
    return result

print("Grounding function ready.")

Grounding function ready.


In [9]:
# =============================================================================
# CELL 9 - Base model demo
# =============================================================================

print("\nBase model demo (before fine-tuning)...")
print("Watch for tool calls firing in the evidence log.\n")

demo_text = (
    "The Great Wall of China is visible from space with the naked eye. "
    "Einstein failed his math exams as a child. "
    "Marie Curie won three Nobel Prizes. "
    "Water boils at 100 degrees Celsius at sea level."
)

demo_result = ground_text(demo_text, verbose=True)

if EVIDENCE_LOG:
    print("\nEvidence calls that fired:")
    for entry in EVIDENCE_LOG:
        print("  ", json.dumps(entry))


Base model demo (before fine-tuning)...
Watch for tool calls firing in the evidence log.

Analyzing: The Great Wall of China is visible from space with the naked eye....

 VeritasLens Analysis
 Overall reliability : 0%
 Claims analyzed     : 4
 Supported           : 0
 Not supported       : 3
 Contested           : 0
 Evidence calls made : 10
 Processing time     : 158.7s
--------------------------------------------------------------
 Evidence retrieved:
  [fetch_evidence] The Great Wall of China is visible from  -> UNSUPPORTED
  [check_knowledge_base] The Great Wall of China is visible from  -> UNSUPPORTED
  [fetch_evidence] Einstein failed his math exams as a chil -> UNSUPPORTED
  [check_knowledge_base] Einstein failed his math exams as a chil -> UNSUPPORTED
  [resolve_entity] Einstein -> 
  [fetch_evidence] Marie Curie won three Nobel Prizes. -> UNSUPPORTED
  [check_knowledge_base] Marie Curie won three Nobel Prizes. -> UNSUPPORTED
  [resolve_entity] Marie Curie -> 
---------------

In [10]:
# =============================================================================
# CELL 10 - Training data
# =============================================================================

def make_verdict(claims, reliability, summary):
    """Serialize a verdict dict to JSON string for training examples."""
    return json.dumps(
        {"claims": claims, "overall_reliability": reliability, "summary": summary},
        indent=2
    )


SEED_EXAMPLES = [

    # Great Wall
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "The Great Wall of China is visible from space with the naked eye, "
                    "as confirmed by multiple astronauts."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
                    "{\"claim\": \"The Great Wall of China is visible from space\", "
                    "\"search_query\": \"Great Wall China visible space naked eye\"}}"
                    "</tool_call>\n"
                    "<tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
                    "{\"claim\": \"The Great Wall is visible from space\", "
                    "\"domain\": \"geography\"}}</tool_call>\n"
                    "<tool_call>{\"name\": \"resolve_entity\", \"arguments\": "
                    "{\"entity_name\": \"Great Wall of China\", \"entity_type\": \"place\", "
                    "\"claimed_attribute\": \"visible from space\"}}</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "The Great Wall of China is visible from space with the naked eye",
                                "verdict": "UNSUPPORTED",
                                "confidence": 0.97,
                                "evidence": ["NASA", "Yang Liwei (China's first astronaut)"],
                                "reasoning": (
                                    "The wall is only 9 meters wide, physically impossible to "
                                    "resolve from orbital altitude. NASA and Yang Liwei confirm this."
                                ),
                                "corrected": (
                                    "The Great Wall is NOT visible from space with the naked eye. "
                                    "This is a well-documented myth."
                                )
                            },
                            {
                                "text": "confirmed by multiple astronauts",
                                "verdict": "UNSUPPORTED",
                                "confidence": 0.95,
                                "evidence": ["NASA", "CNSA"],
                                "reasoning": (
                                    "The opposite is true. Astronauts have stated they cannot "
                                    "see the wall from space."
                                ),
                                "corrected": (
                                    "Astronauts have confirmed they CANNOT see the Great Wall."
                                )
                            }
                        ],
                        reliability=0.0,
                        summary=(
                            "Both claims are hallucinations. The Great Wall is impossible to see "
                            "from orbit and astronauts have said the opposite of what the text claims."
                        )
                    )
                )
            }
        ]
    },

    # Einstein math
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "Albert Einstein famously failed his math exams as a child, "
                    "proving that early struggles do not predict future success."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
                    "{\"claim\": \"Einstein failed his math exams as a child\", "
                    "\"search_query\": \"Einstein math school records childhood\"}}"
                    "</tool_call>\n"
                    "<tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
                    "{\"claim\": \"Einstein failed math\", \"domain\": \"history\"}}"
                    "</tool_call>\n"
                    "<tool_call>{\"name\": \"resolve_entity\", \"arguments\": "
                    "{\"entity_name\": \"Albert Einstein\", \"entity_type\": \"person\", "
                    "\"claimed_attribute\": \"failed math exams\"}}</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "Albert Einstein famously failed his math exams as a child",
                                "verdict": "UNSUPPORTED",
                                "confidence": 0.96,
                                "evidence": ["ETH Zurich records", "Einstein Archive", "Walter Isaacson biography"],
                                "reasoning": (
                                    "Einstein mastered calculus by age 15. The myth comes from "
                                    "misreading Swiss grade scales where 6 is highest, not lowest."
                                ),
                                "corrected": (
                                    "Einstein did NOT fail math. He was exceptional from childhood "
                                    "and mastered calculus by age 15."
                                )
                            },
                            {
                                "text": "proving that early struggles do not predict future success",
                                "verdict": "OPINION",
                                "confidence": 0.90,
                                "evidence": [],
                                "reasoning": "This is an opinion, not a verifiable factual claim.",
                                "corrected": None
                            }
                        ],
                        reliability=0.5,
                        summary=(
                            "The factual claim about Einstein failing math is false. "
                            "The interpretive claim is an opinion."
                        )
                    )
                )
            }
        ]
    },

    # Mixed Curie
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "Marie Curie discovered radium in 1898 and was the first woman to win "
                    "a Nobel Prize. She went on to win three Nobel Prizes in total."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"resolve_entity\", \"arguments\": "
                    "{\"entity_name\": \"Marie Curie\", \"entity_type\": \"person\", "
                    "\"claimed_attribute\": \"number of Nobel prizes\"}}</tool_call>\n"
                    "<tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
                    "{\"claim\": \"Curie three Nobel prizes\", \"domain\": \"history\"}}"
                    "</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "Marie Curie discovered radium in 1898",
                                "verdict": "SUPPORTED",
                                "confidence": 0.99,
                                "evidence": ["Nobel Prize Foundation", "Curie Institute Paris"],
                                "reasoning": "Announced December 1898. Confirmed by historical records.",
                                "corrected": None
                            },
                            {
                                "text": "first woman to win a Nobel Prize",
                                "verdict": "SUPPORTED",
                                "confidence": 0.99,
                                "evidence": ["Nobel Prize Foundation"],
                                "reasoning": "She received the Physics prize in 1903.",
                                "corrected": None
                            },
                            {
                                "text": "She went on to win three Nobel Prizes in total",
                                "verdict": "UNSUPPORTED",
                                "confidence": 0.98,
                                "evidence": ["Nobel Prize Foundation official records"],
                                "reasoning": (
                                    "Marie Curie won exactly two Nobel Prizes: "
                                    "Physics 1903 and Chemistry 1911. No third prize."
                                ),
                                "corrected": (
                                    "Marie Curie won TWO Nobel Prizes, not three: "
                                    "Physics (1903) and Chemistry (1911)."
                                )
                            }
                        ],
                        reliability=0.67,
                        summary=(
                            "Two of three claims are accurate. The text incorrectly states "
                            "three Nobel Prizes when she won two."
                        )
                    )
                )
            }
        ]
    },

    # All correct
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "Water boils at 100 degrees Celsius at standard atmospheric pressure. "
                    "At higher altitudes where air pressure is lower, water boils at a lower temperature."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
                    "{\"claim\": \"Water boils at 100 degrees Celsius at standard pressure\", "
                    "\"search_query\": \"water boiling point 100 Celsius sea level\"}}"
                    "</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "Water boils at 100 degrees Celsius at standard atmospheric pressure",
                                "verdict": "SUPPORTED",
                                "confidence": 1.0,
                                "evidence": ["NIST Chemistry WebBook", "CRC Handbook of Chemistry"],
                                "reasoning": "Fundamental physical constant. 100C at 1 atm standard pressure.",
                                "corrected": None
                            },
                            {
                                "text": "At higher altitudes where air pressure is lower, water boils at a lower temperature",
                                "verdict": "SUPPORTED",
                                "confidence": 1.0,
                                "evidence": ["Basic thermodynamics", "NIST"],
                                "reasoning": "Correct. At 3,000m water boils at approximately 90C.",
                                "corrected": None
                            }
                        ],
                        reliability=1.0,
                        summary="Both claims are scientifically accurate. This text can be trusted."
                    )
                )
            }
        ]
    },

    # Medical misinformation
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "Studies conclusively prove that taking high doses of Vitamin C "
                    "prevents you from catching colds. Humans only use 10 percent of "
                    "their brains, so there is vast untapped cognitive potential."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
                    "{\"claim\": \"Vitamin C prevents colds\", "
                    "\"search_query\": \"vitamin C cold prevention cochrane review\"}}"
                    "</tool_call>\n"
                    "<tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
                    "{\"claim\": \"vitamin c prevents colds\", \"domain\": \"medicine\"}}"
                    "</tool_call>\n"
                    "<tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
                    "{\"claim\": \"humans use 10 percent brain\", \"domain\": \"science\"}}"
                    "</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "Studies conclusively prove that Vitamin C prevents colds",
                                "verdict": "UNSUPPORTED",
                                "confidence": 0.92,
                                "evidence": ["Cochrane Review 2023 (31 trials, 11,306 participants)", "NIH"],
                                "reasoning": (
                                    "The Cochrane Review found no reduction in cold incidence "
                                    "in the general population."
                                ),
                                "corrected": (
                                    "Vitamin C does NOT prevent colds. The Cochrane Review "
                                    "found no preventive effect in 31 trials."
                                )
                            },
                            {
                                "text": "Humans only use 10 percent of their brains",
                                "verdict": "UNSUPPORTED",
                                "confidence": 0.99,
                                "evidence": ["Johns Hopkins Medicine", "fMRI research", "Scientific American"],
                                "reasoning": (
                                    "Brain imaging shows all regions active at some point. "
                                    "No dormant 90 percent region exists."
                                ),
                                "corrected": (
                                    "Humans use virtually all of their brain. "
                                    "This is a well-documented myth."
                                )
                            }
                        ],
                        reliability=0.0,
                        summary=(
                            "Both claims are well-documented hallucinations contradicted "
                            "by strong scientific evidence."
                        )
                    )
                )
            }
        ]
    },

    # Quote misattribution
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "As Albert Einstein famously said, the definition of insanity is "
                    "doing the same thing over and over and expecting different results."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
                    "{\"claim\": \"Einstein definition insanity quote\", \"domain\": \"quotes\"}}"
                    "</tool_call>\n"
                    "<tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
                    "{\"claim\": \"Einstein insanity doing same thing expecting different results\", "
                    "\"search_query\": \"Einstein insanity quote origin source verified\"}}"
                    "</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "Einstein said the definition of insanity is doing the same thing and expecting different results",
                                "verdict": "UNSUPPORTED",
                                "confidence": 0.92,
                                "evidence": [
                                    "Quote Investigator (quoteinvestigator.com)",
                                    "Narcotics Anonymous literature 1981"
                                ],
                                "reasoning": (
                                    "No verified Einstein source exists for this quote. "
                                    "Earliest known appearance is NA literature from 1981, "
                                    "26 years after Einstein died."
                                ),
                                "corrected": (
                                    "This quote is NOT verified as Einstein's. "
                                    "Its earliest known source is Narcotics Anonymous (1981)."
                                )
                            }
                        ],
                        reliability=0.0,
                        summary=(
                            "The quote attributed to Einstein has no verified source. "
                            "It is a misattribution."
                        )
                    )
                )
            }
        ]
    },

    # Contested geographic claim
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "The Amazon River is the longest river in the world and carries "
                    "more freshwater than any other river on Earth."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"resolve_entity\", \"arguments\": "
                    "{\"entity_name\": \"Amazon River\", \"entity_type\": \"place\", "
                    "\"claimed_attribute\": \"longest river\"}}</tool_call>\n"
                    "<tool_call>{\"name\": \"check_knowledge_base\", \"arguments\": "
                    "{\"claim\": \"Amazon longest river\", \"domain\": \"geography\"}}"
                    "</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "The Amazon River is the longest river in the world",
                                "verdict": "CONTESTED",
                                "confidence": 0.75,
                                "evidence": ["USGS", "National Geographic", "INPE Brazil 2008"],
                                "reasoning": (
                                    "The Nile is traditionally measured as longer at 6,650 km. "
                                    "A 2008 Brazilian study using a different source point "
                                    "suggested the Amazon may be longer. Genuinely contested."
                                ),
                                "corrected": (
                                    "Whether the Amazon or Nile is longest is disputed by "
                                    "geographers. The Nile is traditionally longer."
                                )
                            },
                            {
                                "text": "carries more freshwater than any other river on Earth",
                                "verdict": "SUPPORTED",
                                "confidence": 0.99,
                                "evidence": ["USGS Water Resources", "National Geographic"],
                                "reasoning": (
                                    "The Amazon discharges about 20 percent of all freshwater "
                                    "entering the world's oceans."
                                ),
                                "corrected": None
                            }
                        ],
                        reliability=0.5,
                        summary=(
                            "One claim is accurate (freshwater volume), one is genuinely "
                            "contested (longest river). Use with caution."
                        )
                    )
                )
            }
        ]
    },

    # All correct science
    {
        "conversations": [
            {
                "role": "user",
                "content": (
                    "Analyze this text for factual accuracy:\n\n"
                    "The speed of light in a vacuum is approximately 299,792 kilometers "
                    "per second. No information or matter can travel faster than c."
                )
            },
            {
                "role": "assistant",
                "content": (
                    "<tool_call>{\"name\": \"fetch_evidence\", \"arguments\": "
                    "{\"claim\": \"speed of light 299792 km per second\", "
                    "\"search_query\": \"speed of light value NIST\"}}</tool_call>\n\n"
                    + make_verdict(
                        claims=[
                            {
                                "text": "The speed of light in a vacuum is approximately 299,792 kilometers per second",
                                "verdict": "SUPPORTED",
                                "confidence": 1.0,
                                "evidence": ["NIST", "BIPM", "CODATA 2018"],
                                "reasoning": "Exactly 299,792,458 m/s by definition since 1983. Accurate.",
                                "corrected": None
                            },
                            {
                                "text": "No information or matter can travel faster than c",
                                "verdict": "SUPPORTED",
                                "confidence": 0.99,
                                "evidence": ["Special relativity", "Experimental physics"],
                                "reasoning": "Foundational principle of special relativity, confirmed by experiment.",
                                "corrected": None
                            }
                        ],
                        reliability=1.0,
                        summary="Both claims are accurate. This text is fully reliable."
                    )
                )
            }
        ]
    }
]

print("Seed examples:", len(SEED_EXAMPLES))

Seed examples: 8


In [11]:
# =============================================================================
# CELL 11 - Load TruthfulQA
# =============================================================================
def load_truthfulqa(max_examples=200):
    extra = []
    try:
        from datasets import load_dataset

        ds = load_dataset("truthful_qa", "generation", split="validation")
        count = 0
        for row in ds:
            if count >= max_examples:
                break
            answer = row.get("best_answer", "")
            question = row.get("question", "")
            if not answer or len(answer) < 20:
                continue

            claim = answer[:80].replace('"', "'")
            search_query = " ".join(answer.split()[:5])

            # FIX: require TWO tool calls so model learns multi-round is normal
            fetch_call = json.dumps({
                "name": "fetch_evidence",
                "arguments": {"claim": claim, "search_query": search_query}
            })
            kb_call = json.dumps({
                "name": "check_knowledge_base",
                "arguments": {"claim": claim, "domain": "general"}
            })

            extra.append({
                "conversations": [
                    {
                        "role": "user",
                        "content": "Analyze this text for factual accuracy:\n\n{}".format(answer[:200])
                    },
                    {
                        "role": "assistant",
                        "content": (
                            "<tool_call>{}</tool_call>\n".format(fetch_call) +
                            "<tool_call>{}</tool_call>\n".format(kb_call) +
                            make_verdict(
                                claims=[{
                                    "text": answer[:150],
                                    "verdict": "SUPPORTED",
                                    "confidence": 0.85,
                                    "evidence": ["TruthfulQA verified answer", "Wikipedia"],
                                    "reasoning": "Verified accurate answer from TruthfulQA benchmark.",
                                    "corrected": None
                                }],
                                reliability=0.85,
                                summary="This response is factually grounded per TruthfulQA."
                            )
                        )
                    }
                ]
            })
            count += 1

        print("TruthfulQA examples loaded:", len(extra))
    except Exception as e:
        print("TruthfulQA unavailable ({}). Using seed examples only.".format(e))
    return extra

extra_examples = load_truthfulqa(max_examples=200)
all_examples = SEED_EXAMPLES + extra_examples
random.shuffle(all_examples)
print("Total training examples:", len(all_examples))

TruthfulQA examples loaded: 200
Total training examples: 208


In [12]:
# =============================================================================
# CELL 12 - Prepare dataset
# =============================================================================

from datasets import Dataset


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        formatted = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        if formatted.startswith("<bos>"):
            formatted = formatted[len("<bos>"):]
        texts.append(formatted)
    return {"text": texts}


raw_dataset  = Dataset.from_list([{"conversations": ex["conversations"]} for ex in all_examples])
train_dataset = raw_dataset.map(formatting_prompts_func, batched=True)

print("Training dataset:", len(train_dataset), "examples")
print("\nSample (first 400 chars):")
print(train_dataset[0]["text"][:400])

Map:   0%|          | 0/208 [00:00<?, ? examples/s]

Training dataset: 208 examples

Sample (first 400 chars):
<|turn>user
Analyze this text for factual accuracy:

As Albert Einstein famously said, the definition of insanity is doing the same thing over and over and expecting different results.<turn|>
<|turn>model
<tool_call>{"name": "check_knowledge_base", "arguments": {"claim": "Einstein definition insanity quote", "domain": "quotes"}}</tool_call>
<tool_call>{"name": "fetch_evidence", "arguments": {"clai


In [13]:
# =============================================================================
# CELL 13 - Attach LoRA adapters
# =============================================================================

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    random_state=3407
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print("Trainable: {:,} / {:,} ({:.2%})".format(trainable, total, trainable / total))

Trainable: 18,350,080 / 5,997,636,128 (0.31%)


In [14]:
# =============================================================================
# CELL 14 - Train
# =============================================================================

from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,               # set num_train_epochs=1 for full run
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="./veritaslens-checkpoints",
        report_to="none",
        save_strategy="no"
    )
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n"
)

print("Training...")
trainer_stats = trainer.train()
print("Training loss: {:.4f}".format(trainer_stats.training_loss))
print("Runtime: {:.1f} min".format(trainer_stats.metrics.get("train_runtime", 0) / 60))

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/208 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/208 [00:00<?, ? examples/s]

Filter (num_proc=12):   0%|          | 0/208 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 208 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,350,080 of 8,014,506,528 (0.23% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
5,13.971974
10,12.072849
15,5.286427
20,3.242357
25,2.247018
30,1.676336
35,1.410050
40,0.832880
45,0.892587
50,0.611496


Unsloth: Will smartly offload gradients to save VRAM!
Training loss: 3.6244
Runtime: 4.7 min


In [15]:
# =============================================================================
# CELL 15 - Evaluation
# =============================================================================
FastModel.for_inference(model)

EVAL_SUITE = [
    {"text": "The Great Wall of China is visible from space with the naked eye.",
     "expected": "UNSUPPORTED", "category": "geography"},
    {"text": "Albert Einstein failed his math exams as a child.",
     "expected": "UNSUPPORTED", "category": "biography"},
    {"text": "Marie Curie won three Nobel Prizes.",
     "expected": "UNSUPPORTED", "category": "history"},
    {"text": "Studies conclusively prove that Vitamin C prevents colds.",
     "expected": "UNSUPPORTED", "category": "medicine"},
    {"text": "Humans only use 10 percent of their brains.",
     "expected": "UNSUPPORTED", "category": "neuroscience"},
    {"text": "As Einstein said, the definition of insanity is doing the same thing over and over.",
     "expected": "UNSUPPORTED", "category": "quotes"},
    {"text": "Water boils at 100 degrees Celsius at sea level.",
     "expected": "SUPPORTED", "category": "science"},
    {"text": "The Amazon River carries more freshwater than any other river on Earth.",
     "expected": "SUPPORTED", "category": "geography"},
    # Generalization cases: NOT in the KB, require Wikipedia fallback
    {"text": "The Moon is the largest natural satellite in the Solar System.",
     "expected": "UNSUPPORTED", "category": "astronomy"},
    {"text": "Penicillin was discovered by Alexander Fleming in 1928.",
     "expected": "SUPPORTED", "category": "history"},
]

def run_evaluation():
    print("\n" + "=" * 66)
    print(" Evaluation Suite (post fine-tuning)")
    print("=" * 66)
    correct = 0
    results = []
    for case in EVAL_SUITE:
        result = ground_text(case["text"], verbose=False)

        # Pick the most decisive verdict across all claims
        priority = {"UNSUPPORTED": 0, "SUPPORTED": 1, "CONTESTED": 2,
                    "UNVERIFIABLE": 3, "OPINION": 4}
        detected = "UNVERIFIABLE"
        for claim in result.claims:
            v = claim.get("verdict", "")
            if v in priority and priority[v] < priority.get(detected, 99):
                detected = v

        kb_fired = any(
            e.get("tool") in ("fetch_evidence", "check_knowledge_base")
            for e in result.tools_called
        )
        wiki_fired = any(
            e.get("method") == "wikipedia"
            for e in result.tools_called
        )

        passed = detected == case["expected"]
        if passed:
            correct += 1
        status = "PASS" if passed else "FAIL"
        print("  [{}] {:<14} expected={:<13} got={:<13} kb={} wiki={}".format(
            status,
            case["category"],
            case["expected"],
            detected,
            "yes" if kb_fired else "no",
            "yes" if wiki_fired else "no"
        ))
        results.append({
            "category": case["category"],
            "expected": case["expected"],
            "detected": detected,
            "passed": passed,
            "kb_called": kb_fired,
            "wiki_called": wiki_fired
        })

    print("-" * 66)
    print("  Accuracy: {}/{} = {:.0%}".format(correct, len(EVAL_SUITE),
                                               correct / len(EVAL_SUITE)))
    print("=" * 66)
    return results, correct / len(EVAL_SUITE)

eval_results, eval_accuracy = run_evaluation()


 Evaluation Suite (post fine-tuning)
  [PASS] geography      expected=UNSUPPORTED   got=UNSUPPORTED   kb=no wiki=no
  [PASS] biography      expected=UNSUPPORTED   got=UNSUPPORTED   kb=yes wiki=no
  [PASS] history        expected=UNSUPPORTED   got=UNSUPPORTED   kb=yes wiki=no
  [PASS] medicine       expected=UNSUPPORTED   got=UNSUPPORTED   kb=yes wiki=no
  [PASS] neuroscience   expected=UNSUPPORTED   got=UNSUPPORTED   kb=no wiki=no
  [PASS] quotes         expected=UNSUPPORTED   got=UNSUPPORTED   kb=yes wiki=no
  [PASS] science        expected=SUPPORTED     got=SUPPORTED     kb=no wiki=no
  [PASS] geography      expected=SUPPORTED     got=SUPPORTED     kb=no wiki=no
  [FAIL] astronomy      expected=UNSUPPORTED   got=UNVERIFIABLE  kb=yes wiki=no
  [PASS] history        expected=SUPPORTED     got=SUPPORTED     kb=no wiki=no
------------------------------------------------------------------
  Accuracy: 9/10 = 90%


In [16]:
# =============================================================================
# CELL 16 - Interpretability: token attribution heatmap
# =============================================================================
# Method: causal mediation analysis.
# We ask the model "is this claim true or false?" and measure which input
# tokens, when replaced with an unknown token, cause the biggest shift in
# the true-vs-false logit difference.
#
# This identifies which words are CAUSALLY important to the verdict,
# not just which words the model attended to.

def compute_token_attribution(claim_text, n_tokens=12):
    """
    Compute causal attribution scores for each token in the claim.
    Returns tokens and normalized attribution scores.
    """
    prompt = "Is the following claim true or false? Reply with one word: true or false.\n{}".format(claim_text)
    messages = [{"role": "user", "content": prompt}]

    prompt_str = raw_tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs     = raw_tok(prompt_str, return_tensors="pt").to(model.device)
    input_ids  = inputs["input_ids"]
    input_ids_cast = input_ids.to(dtype=torch.long)

    # Get token IDs for "true" and "false"
    true_id  = raw_tok.encode("true",  add_special_tokens=False)[-1]
    false_id = raw_tok.encode("false", add_special_tokens=False)[-1]
    unk_id   = raw_tok.unk_token_id or 3

    # Baseline: run model on unmodified input
    with torch.inference_mode():
        out = model(input_ids=input_ids_cast, return_dict=True)

    logits = out.logits[0, -1].float()
    baseline_score = (logits[true_id] - logits[false_id]).item()

    # Get the claim tokens (last n_tokens of the input)
    claim_ids  = raw_tok.encode(claim_text, add_special_tokens=False)[:n_tokens]
    claim_strs = raw_tok.convert_ids_to_tokens(claim_ids)
    total_len  = input_ids.shape[1]
    claim_start = max(0, total_len - len(claim_ids))

    attributions = []
    for i in range(len(claim_ids)):
        patched = input_ids_cast.clone()
        pos = claim_start + i
        if pos < total_len:
            patched[0, pos] = unk_id

        with torch.inference_mode():
            p_out = model(input_ids=patched, return_dict=True)

        p_score = (
            p_out.logits[0, -1, true_id].float()
            - p_out.logits[0, -1, false_id].float()
        ).item()

        attributions.append(abs(baseline_score - p_score))

    max_a = max(attributions) if attributions else 1.0
    if max_a == 0:
        max_a = 1.0
    normalized = [round(a / max_a, 4) for a in attributions]

    # Clean token strings: remove SentencePiece word-boundary marker (ordinal 9601)
    clean = ["".join(ch for ch in t if ord(ch) != 9601) for t in claim_strs]

    # Filter out punctuation-only tokens from top-5 to get semantic tokens
    semantic = [
        (tok, score) for tok, score in zip(clean, normalized)
        if tok.strip(".,!?;:") != ""
    ]
    top_5 = sorted(semantic, key=lambda x: x[1], reverse=True)[:5]

    return {
        "claim":        claim_text,
        "tokens":       clean,
        "attributions": normalized,
        "top_5":        [{"token": t, "score": s} for t, s in top_5],
        "interpretation": "Top influential tokens: " + ", ".join(
            "'{}' ({:.2f})".format(t, s) for t, s in top_5[:3]
        )
    }


print("\nInterpretability Analysis")
print("Claim: 'The Great Wall of China is visible from space with the naked eye.'")
print()

attr = compute_token_attribution(
    "The Great Wall of China is visible from space with the naked eye."
)

print("Token attribution (higher = more causally influential):")
print()
for tok, score in zip(attr["tokens"], attr["attributions"]):
    bar = "#" * int(score * 30)
    print("  {:<14} {} {:.3f}".format(tok, bar, score))

print()
print(attr["interpretation"])
print()
print("Note: Punctuation tokens filtered from top-5. Semantic tokens shown.")


Interpretability Analysis
Claim: 'The Great Wall of China is visible from space with the naked eye.'

Token attribution (higher = more causally influential):

  The            #### 0.149
  Great          # 0.049
  Wall           ### 0.100
  of             ### 0.124
  China          # 0.055
  is             ### 0.122
  visible        ####### 0.251
  from            0.017
  space           0.030
  with           ### 0.126
  the            ############################## 1.000
  naked          ######################## 0.814

Top influential tokens: 'the' (1.00), 'naked' (0.81), 'visible' (0.25)

Note: Punctuation tokens filtered from top-5. Semantic tokens shown.


In [17]:
# =============================================================================
# CELL 17 - Save LoRA adapter
# =============================================================================

ADAPTER_PATH = "./veritaslens-adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("LoRA adapter saved:", ADAPTER_PATH)
for f in sorted(Path(ADAPTER_PATH).iterdir()):
    print("  {} ({:.0f} KB)".format(f.name, f.stat().st_size / 1024))

LoRA adapter saved: ./veritaslens-adapter
  README.md (5 KB)
  adapter_config.json (2 KB)
  adapter_model.safetensors (71764 KB)
  chat_template.jinja (1 KB)
  processor_config.json (2 KB)
  tokenizer.json (31416 KB)
  tokenizer_config.json (3 KB)


In [18]:
# =============================================================================
# CELL 18 - Export GGUF for Ollama
# =============================================================================

GGUF_DIR = "./veritaslens-gguf"

print("Exporting GGUF (q4_k_m)... this takes about 15 minutes.")

model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

print("GGUF exported:", GGUF_DIR)
print()
print("To deploy locally:")
print("  ollama create veritaslens -f ./veritaslens-gguf_gguf/Modelfile")
print("  ollama run veritaslens")

Exporting GGUF (q4_k_m)... this takes about 15 minutes.
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `./veritaslens-gguf`: 100%|██████████| 1/1 [01:03<00:00, 63.40s/it]


Successfully copied all 1 files from cache to `./veritaslens-gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

Splitting model.safetensors (size: 14.89 GB)...


Unsloth: Merging weights into 16bit: 100%|██████████| 9/9 [01:52<00:00, 12.54s/it]


Unsloth: Regenerating safetensors index...
Unsloth: Merge process complete. Saved to `/content/veritaslens-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['./veritaslens-gguf_gguf/gemma-4-E4B-it.F16.gguf', './veritaslens-gguf_gguf/gemma-4-E4B-it.F16-mmproj.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['./veritaslens-gguf_gguf/gemma-4-E4B-it.Q4_K_M.gguf', './veritaslens-gguf_gguf/gemma-4-E4B-it.F16-mmproj.gguf']


Unsloth: example usage for Multimodal LLMs: /root/.unsloth/llama.cpp/llama-mtmd-cli -m ./veritaslens-gguf_gguf/gemma-4-E4B-it.Q4_K_M.gguf --mmproj ./veritaslens-gguf_gguf/gemma-4-E4B-it.F16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image
Unsloth: Saved Ollama Modelfile to ./veritaslens-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f ./veri

In [19]:
# =============================================================================
# CELL 19 - Gradio demo
# =============================================================================

try:
    import gradio as gr
    GRADIO_OK = True
except ImportError:
    GRADIO_OK = False
    print("pip install gradio")

VERDICT_MARKER = {
    "SUPPORTED":    "[OK]",
    "UNSUPPORTED":  "[!!]",
    "CONTESTED":    "[??]",
    "UNVERIFIABLE": "[--]",
    "OPINION":      "[~~]"
}

VERDICT_LABEL = {
    "SUPPORTED":    "Supported by evidence",
    "UNSUPPORTED":  "Not supported -- likely hallucination",
    "CONTESTED":    "Contested -- sources disagree",
    "UNVERIFIABLE": "Cannot verify with available sources",
    "OPINION":      "Opinion, not a factual claim"
}

EXAMPLES = [
    "The Great Wall of China is visible from space. Einstein failed math as a child. Marie Curie won three Nobel Prizes.",
    "Water boils at 100 degrees Celsius at sea level. The Amazon carries more freshwater than any other river.",
    "Studies prove Vitamin C prevents colds. Humans only use 10 percent of their brains.",
    "As Einstein famously said, the definition of insanity is doing the same thing and expecting different results.",
]


def analyze_for_gradio(input_text):
    if not input_text or not input_text.strip():
        return "Please enter some text to analyze."

    try:
        result = ground_text(input_text, verbose=False)
    except Exception as e:
        return "Analysis error: {}".format(str(e))

    if not result.claims:
        return "No verifiable claims detected."

    lines = [
        "## VeritasLens Results",
        "",
        "**Overall reliability: {:.0%}** | Claims: {} | Time: {:.1f}s | Evidence calls: {}".format(
            result.overall_reliability,
            len(result.claims),
            result.processing_time_s,
            len(result.tools_called)
        ),
        ""
    ]

    # Show evidence calls that actually fired
    if result.tools_called:
        lines.append("**Evidence retrieved:**")
        for call in result.tools_called[:8]:
            lines.append("- `{}`: {} -> {}".format(
                call.get("tool", ""),
                call.get("claim", call.get("entity", ""))[:45],
                call.get("verdict_signal", call.get("verdict", ""))
            ))
        lines.append("")

    lines.append("---")

    for i, claim in enumerate(result.claims, 1):
        verdict   = claim.get("verdict", "UNVERIFIABLE")
        conf      = claim.get("confidence", 0)
        text      = claim.get("text", "")
        reasoning = claim.get("reasoning", "")
        corrected = claim.get("corrected")
        evidence  = claim.get("evidence", [])

        marker = VERDICT_MARKER.get(verdict, "[?]")
        label  = VERDICT_LABEL.get(verdict, verdict)

        lines.append("")
        lines.append("### {} Claim {} -- {:.0%} confidence".format(marker, i, conf))
        lines.append("**{}**".format(label))
        lines.append("")
        lines.append("*\"{}\"*".format(text))
        lines.append("")
        lines.append(reasoning)

        if evidence:
            lines.append("")
            lines.append("**Sources:** " + " | ".join(str(e) for e in evidence[:3]))

        if corrected:
            lines.append("")
            lines.append("**Corrected:** {}".format(corrected))

        lines.append("")
        lines.append("---")

    lines.append("")
    lines.append("*VeritasLens -- Gemma 4 Good Hackathon 2026 | Safety and Trust Track*")

    return "\n".join(lines)


if GRADIO_OK:
    with gr.Blocks(title="VeritasLens -- Hallucination Detector") as demo:

        gr.Markdown(
            "# VeritasLens\n"
            "**Real-Time Hallucination Detection powered by Gemma 4**\n\n"
            "Paste any AI-generated text and click Analyze. "
            "VeritasLens retrieves evidence for every claim and returns "
            "a structured verdict. Analysis takes 1-3 minutes on T4 GPU."
        )

        text_input = gr.Textbox(
            label="AI-generated text to analyze",
            placeholder="Paste any AI-generated text here...",
            lines=6
        )

        analyze_btn = gr.Button("Analyze", variant="primary", size="lg")

        output_md = gr.Markdown(
            value="Results will appear here after you click Analyze.",
            label="Analysis"
        )

        gr.Examples(
            examples=[[ex] for ex in EXAMPLES],
            inputs=[text_input],
            label="Quick examples -- click to load, then click Analyze"
        )

        gr.Markdown(
            "---\n"
            "Gemma 4 Good Hackathon 2026 | Safety and Trust Track | "
            "Main Track | Unsloth | CC-BY 4.0"
        )

        analyze_btn.click(
            fn=analyze_for_gradio,
            inputs=[text_input],
            outputs=[output_md],
            show_progress="full"
        )

    demo.launch(share=True)

else:
    print("Gradio not available. Run: result = ground_text('your text here')")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d25ab6c3899b9f93de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
# =============================================================================
# Summary
# =============================================================================
#
# What this notebook demonstrates:
#
#   Component                   What it does
#   ----------------------------------------------------------
#   Gemma 4 E4B via Unsloth     Core language model for claim analysis
#   Offline knowledge base      12 hallucination categories, instant recall
#   Evidence retrieval          Fires on every claim, logged in outputs
#   Unsloth QLoRA fine-tuning   Domain-specific hallucination detection
#   train_on_responses_only     Efficient training on verdict outputs only
#   GGUF export                 Runs on any laptop via Ollama, no cloud
#   Token attribution           Causal mediation, semantic tokens only
#   Gradio demo                 Shows evidence calls and verdicts live
#
#   Prize targets:
#     Safety & Trust  $10K     Transparent, explainable, grounded AI
#     Main Track      $50K     Technical depth + real-world impact
#     Unsloth         $10K     Fine-tuned Gemma 4 E4B for specific task
#
# =============================================================================